# Train the dengue model

Load the monthly table, drop incomplete rows, and add `year`, `month_num`, and `log_cases` so the data is ready for modelling.


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

TRAIN_END_YEAR = 2019
INPUT_FILE = "processed/dengue_monthly.csv"
FALLBACK_INPUT_FILE = "data_used/dengue_monthly.csv"
OUTPUT_FILE = "data_used/dengue_model.json"

try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"{INPUT_FILE} not found, loading {FALLBACK_INPUT_FILE}")
    df = pd.read_csv(FALLBACK_INPUT_FILE)

print("Rows loaded:", len(df))
df = df.dropna(subset=["rain_prev_departure", "cases"]).copy()
print("Rows after dropping missing cases/departure:", len(df))

df["year"] = df["month"].str[:4].astype(int)
df["month_num"] = df["month"].str[5:7].astype(int)
df["log_cases"] = np.log1p(df["cases"])

print("\nRows per year:")
print(df.groupby("year").size().to_string())
print("\nRows per state:")
print(df.groupby("state").size().sort_values(ascending=False).to_string())


## 1. Time split

Use years up to `TRAIN_END_YEAR` for training and later years for testing. Warn if the test set is very small.


In [ ]:
train = df[df["year"] <= TRAIN_END_YEAR].copy()
test = df[df["year"] > TRAIN_END_YEAR].copy()

print("Train rows:", len(train))
print("Test rows: ", len(test))
if len(train) > 0:
    print("Train years:", int(train["year"].min()), "to", int(train["year"].max()))
if len(test) > 0:
    print("Test years: ", int(test["year"].min()), "to", int(test["year"].max()))
if len(test) < 30:
    print("WARNING: The test set has fewer than 30 rows. Lower TRAIN_END_YEAR to create a larger test set.")


## 2. Baseline seasonal average

From training rows only, compute the mean `log_cases` for each state-month pair. Fall back to the month average across states, then to the overall training mean.


In [ ]:
overall_mean = train["log_cases"].mean()
seasonal_table = (
    train.groupby(["state", "month_num"])["log_cases"]
    .mean()
    .rename("mean_log_cases")
    .reset_index()
)
month_fallback = train.groupby("month_num")["log_cases"].mean()

def baseline_predict(frame):
    predictions = frame[["state", "month_num"]].merge(
        seasonal_table, on=["state", "month_num"], how="left"
    )["mean_log_cases"]
    predictions.index = frame.index
    month_means = frame["month_num"].map(month_fallback)
    return predictions.fillna(month_means).fillna(overall_mean)

train["baseline_pred"] = baseline_predict(train)
test["baseline_pred"] = baseline_predict(test)

baseline_mae_log = np.mean(np.abs(test["log_cases"] - test["baseline_pred"]))
baseline_mae_cases = np.mean(np.abs(test["cases"] - np.expm1(test["baseline_pred"])))

print("Baseline MAE on log_cases:", baseline_mae_log)
print("Baseline MAE on cases:", baseline_mae_cases)
print("Seasonal table rows:", len(seasonal_table))
print("Month fallback entries:", len(month_fallback))
print("Overall training mean:", overall_mean)


## 3. Rainfall versus baseline residual

Plot training rows only: previous-month rainfall departure on the x-axis and `log_cases` minus the baseline prediction on the y-axis.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.scatter(
    train["rain_prev_departure"],
    train["log_cases"] - train["baseline_pred"],
    alpha=0.5,
    s=15,
)
plt.xlabel("Previous-month rainfall departure")
plt.ylabel("log_cases minus baseline prediction")
plt.title("Training residuals vs previous-month rainfall departure")
plt.tight_layout()
plt.show()


## 4. Weather adjustment

Fit a linear regression on training residuals using clipped previous-month rainfall departure, then add the fitted adjustment to the baseline prediction.


In [ ]:
CLIP_MIN, CLIP_MAX = -1.0, 2.0
train["rain_prev_clipped"] = train["rain_prev_departure"].clip(CLIP_MIN, CLIP_MAX)
test["rain_prev_clipped"] = test["rain_prev_departure"].clip(CLIP_MIN, CLIP_MAX)

residual = train["log_cases"] - train["baseline_pred"]
weather = LinearRegression()
weather.fit(train[["rain_prev_clipped"]], residual)
slope = float(weather.coef_[0])
intercept = float(weather.intercept_)

train["adjusted_pred"] = train["baseline_pred"] + intercept + slope * train["rain_prev_clipped"]
test["adjusted_pred"] = test["baseline_pred"] + intercept + slope * test["rain_prev_clipped"]

print("Slope:", slope)
print("Intercept:", intercept)
print("Training rows used:", len(train))
print(test[["state", "month", "log_cases", "baseline_pred", "adjusted_pred"]].head())


## 5. Compare both methods on the test set

Report MAE on `log_cases` and on the original `cases` scale, then show how many test rows each state contributes.


In [ ]:
adjusted_mae_log = np.mean(np.abs(test["log_cases"] - test["adjusted_pred"]))
adjusted_mae_cases = np.mean(np.abs(test["cases"] - np.expm1(test["adjusted_pred"])))

print("Baseline:         MAE log_cases = {:.4f}, MAE cases = {:.2f}".format(baseline_mae_log, baseline_mae_cases))
print("Weather-adjusted: MAE log_cases = {:.4f}, MAE cases = {:.2f}".format(adjusted_mae_log, adjusted_mae_cases))
print("\nTest rows per state:")
print(test.groupby("state").size().sort_values(ascending=False).to_string())

if baseline_mae_log <= adjusted_mae_log:
    better = "baseline"
    log_difference = adjusted_mae_log - baseline_mae_log
else:
    better = "weather-adjusted"
    log_difference = baseline_mae_log - adjusted_mae_log
print(f"\nOn log_cases, the {better} method is better by {log_difference:.4f} MAE.")


## 6. Save the model

Write the seasonal table, fallbacks, regression coefficients, clip bounds, training cutoff, and test errors to plain JSON.


In [ ]:
seasonal_records = []
for row in seasonal_table.itertuples(index=False):
    seasonal_records.append({
        "state": row.state,
        "month_num": int(row.month_num),
        "mean_log_cases": float(row.mean_log_cases),
    })

model = {
    "seasonal_table": seasonal_records,
    "month_fallback": {str(int(k)): float(v) for k, v in month_fallback.items()},
    "overall_mean": float(overall_mean),
    "slope": slope,
    "intercept": intercept,
    "clip_bounds": [CLIP_MIN, CLIP_MAX],
    "train_end_year": int(TRAIN_END_YEAR),
    "test_errors": {
        "baseline": {
            "mae_log_cases": float(baseline_mae_log),
            "mae_cases": float(baseline_mae_cases),
        },
        "weather_adjusted": {
            "mae_log_cases": float(adjusted_mae_log),
            "mae_cases": float(adjusted_mae_cases),
        },
    },
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(model, f, indent=2)

print("Saved model to:", OUTPUT_FILE)
print("Seasonal table rows:", len(model["seasonal_table"]))
print("Month fallback entries:", len(model["month_fallback"]))
print("Test errors:", model["test_errors"])
